# Offline replay and audit

This notebook needs **no Nebius API key, network connection, or Token Factory credits**. It reads sanitized measured JSON artifacts and calls only deterministic InferDoc verification and hashing code.

In [ ]:
import copy
import json
from pathlib import Path
from inferdoc.evidence.models import EvidenceBundle
from inferdoc.evidence.provenance import canonical_sha256
from inferdoc.experiments.models import ExperimentSpec
from inferdoc.storage import ArtifactStore
from inferdoc.verification import verify

demo_dir = Path("examples/demo_artifacts")
store = ArtifactStore(demo_dir.parent)
integrity = store.verify_manifest(demo_dir.name)
print("manifest integrity:", integrity.model_dump())
assert integrity.valid

## Load the measured baseline, candidate, and admitted experiment

In [ ]:
baseline = EvidenceBundle.model_validate_json((demo_dir / "baseline.json").read_text(encoding="utf-8"))
candidate = EvidenceBundle.model_validate_json((demo_dir / "candidate.json").read_text(encoding="utf-8"))
experiment = ExperimentSpec.model_validate_json((demo_dir / "experiment.json").read_text(encoding="utf-8"))
print({"baseline_run_id": baseline.run_id, "candidate_run_id": candidate.run_id,
       "experiment_id": experiment.id, "changed_variables": experiment.changed_variables})

## Replay Python verification

Relative changes are measured for this captured run. PASS, FAIL, and INCONCLUSIVE remain possible for future reruns.

In [ ]:
report = verify(baseline=baseline, candidate=candidate, experiment=experiment)
print({"status": report.status,
       "metric_results": [item.model_dump(mode="json") for item in report.metric_results],
       "constraint_results": [item.model_dump(mode="json") for item in report.constraint_results]})

## In-memory tamper demonstration

Change only an in-memory copy. The fixture files and manifest stay untouched.

In [ ]:
manifest = json.loads((demo_dir / "manifest.json").read_text(encoding="utf-8"))
original = baseline.model_dump(mode="json")
tampered = copy.deepcopy(original)
tampered["aggregate"]["total_output_tokens"] += 1
expected_hash = manifest["artifacts"]["baseline.json"]["sha256"]
print({"original_matches_manifest": canonical_sha256(original) == expected_hash,
       "tampered_matches_manifest": canonical_sha256(tampered) == expected_hash})
assert canonical_sha256(original) == expected_hash
assert canonical_sha256(tampered) != expected_hash